# Italy–Ghana PINN Pipeline — Colab runner (MPhil thesis extension)

Extends the course-project notebook (`ghana_pinn_colab-notebook.ipynb`) with the
Italy benchmark arm, the SEIQHRS synthetic hospitalisation ablation, the
sparsification axis, and the equity/statistical-testing stages.

**Nothing about the validated engine changes.** `pinn.py`, `prepare_ghana.py`,
`run_ghana.py`, `run_stages.py`, `run_parallel.py`, and `extract_rt.py` are
reused exactly as validated against Millevoi's Table 4. This notebook adds:
`prepare_italy.py`, `synth_hospitalisation.py`, `sparsify.py`, `run_italy.py`,
`equity_metrics.py`, `stat_tests.py`.

## Runtime choice — same reasoning as before, now doubly true

**Pick a high-vCPU CPU runtime, not a GPU one.** The networks are tiny
(4×50 / 4×100 units), so a GPU mostly pays kernel-launch overhead. The Italy
arm adds a second country and a sparsification axis — more independent fits,
not bigger ones — so the "many small CPU workers" strategy scales even better
here than in the Ghana-only course project.

| Colab tier | What to pick | Why |
|---|---|---|
| Free | Runtime → Change runtime type → CPU (default) | 2 vCPUs; run split-only grids, background execution NOT available on free tier so keep the tab open or use Stage-by-stage checkpointing (Section 6) |
| Free + occasional GPU quota | Ignore the GPU, still pick CPU | Benchmark cell below proves it; GPU quota is better saved for something else |
| Pay-as-you-go / Pro | High-RAM CPU runtime, enable Background execution | 8+ vCPUs, survives a closed tab, an 8–20h combined grid actually finishes |
| Pro+ | High-RAM CPU, Background execution, consider 2 parallel sessions | Use `--shard`/`--of` in `run_parallel.py` / `run_italy.py` to split the joint grid across two notebook sessions (Section 6.3) |

If you only have the **free tier**, do not attempt the full joint grid for
both countries in one sitting — follow the staged/sharded plan in Section 6;
each stage is sized to fit inside a free-tier session (~90 min before likely
disconnect) or to checkpoint cleanly if it does not.

## 1. Runtime check

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
import os, platform, subprocess, torch
ncpu = os.cpu_count()
print("python  :", platform.python_version())
print("torch   :", torch.__version__)
print("vCPUs   :", ncpu)
print("cuda    :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout.split("\n")[1])

is_free_tier = ncpu <= 2
if is_free_tier:
    print("\n== Free-tier CPU detected (<=2 vCPUs) ==")
    print("Background execution is unlikely to be available. Use the staged")
    print("checkpointed plan in Section 6 rather than the single big grid call.")
elif ncpu < 8:
    print("\n!! Only", ncpu, "vCPUs. Runtime -> Change runtime type -> High-RAM CPU")
    print("   for more cores if on a paid tier. This matters far more than a GPU.")
else:
    print("\nGood: >=8 vCPUs. The full combined grid is feasible in one sitting.")


python  : 3.12.1
torch   : 2.14.0+cu130
vCPUs   : 4
cuda    : False 
Mem:              15           3           0           0          12          11

!! Only 4 vCPUs. Runtime -> Change runtime type -> High-RAM CPU
   for more cores if on a paid tier. This matters far more than a GPU.


## 2. Upload the project

Upload the extended pipeline archive (`thesis_pipeline.tar.gz` or an equivalent
`.zip` containing `src/` and `data/`). This must include BOTH the original
course-project files and the six new scripts (`prepare_italy.py`,
`synth_hospitalisation.py`, `sparsify.py`, `run_italy.py`, `equity_metrics.py`,
`stat_tests.py`), plus the raw Ghana HERA CSV and the raw Italy ISS
infections+hospitalisations CSV under `data/raw/`.

In [ ]:
# from google.colab import files
# import tarfile, zipfile, os

# os.chdir("/content")
# up = files.upload()
# archive_name = next(k for k in up if k.endswith((".zip", ".tar.gz", ".tgz")))

# if archive_name.endswith(".zip"):
#     zipfile.ZipFile(archive_name).extractall(".")
# else:
#     tarfile.open(archive_name).extractall(".")

# # Adjust this if your archive extracts to a different top-level folder name.
# candidates = [d for d in os.listdir(".") if os.path.isdir(d) and d not in (".config", "sample_data")]
# project_dir = "proj" if "proj" in candidates else (candidates[0] if candidates else ".")
# os.chdir(f"/content/{project_dir}")
# print("cwd:", os.getcwd())
# !pip -q install scipy
# !ls src data 2>/dev/null || ls


Saving thesis-pipeline.zip to thesis-pipeline.zip
cwd: /content/thesis-pipeline
data:
processed  raw

src:
build_italy_national.py    fig_dataquality_v1.py  run_parallel.py
checkpoint_grid_runner.py  pinn.py		  run_stages_italy_arms.py
equity_metrics.py	   prepare_ghana.py	  run_stages.py
equity_metrics_v1.py	   prepare_italy.py	  sparsify.py
extract_rt.py		   prepare_italy_v1.py	  stat_tests.py
extract_rt_v1.py	   run_ghana.py		  stat_tests_v1.py
fig_dataquality.py	   run_italy.py		  synth_hospitalisation.py


## 3. Benchmark: CPU vs GPU (both countries share one engine, so one check suffices)

Same measurement-not-assumption approach as the course project. Reuses the
Ghana daily-regime timing; Italy fits are cheaper per epoch since collocation
count is fixed regardless of series length, so if CPU wins here it wins there
too.

In [3]:
import time, torch, numpy as np, pandas as pd, sys
sys.path.insert(0, "src")

from pinn import Config, ReducedSIRPINN

m = pd.read_csv("data/processed/ghana_national_master.csv", index_col=0, parse_dates=True)
I = m.I_proxy.to_numpy(float); scale = I.max(); origin = 180
td = torch.tensor(np.arange(origin) / origin, dtype=torch.float32).reshape(-1, 1)
yo = torch.tensor(I[:origin] / scale, dtype=torch.float32).reshape(-1, 1)

for dev in (["cpu", "cuda"] if torch.cuda.is_available() else ["cpu"]):
    c = Config(epochs_joint=60, n_collocation=6000, device=dev)
    t0 = time.time(); ReducedSIRPINN(c, tf=origin, scale_I=scale).fit_joint(td, yo)
    dt = time.time() - t0
    print(f"{dev:5s}: {dt:6.1f}s for 60 joint epochs -> "
          f"full 5000-epoch fit ~= {dt/60*5000/60:5.1f} min")


cpu  :   25.9s for 60 joint epochs -> full 5000-epoch fit ~=  35.9 min


## 4. Validation gate — Millevoi Case 4 before all country arms

Before running Ghana, Italy-primary, or Italy-secondary experiments, validate the shared `ReducedSIRPINN` implementation against Millevoi et al.’s Case 4 benchmark. Every country arm uses the same core PINN class, so a failure here would invalidate all downstream country-specific results.

Published Case 4 targets from Millevoi et al.’s Table 4 are:

| Training approach | Published relative error: \(I(t)\) | Published relative error: \(R_t\) |
|---|---:|---:|
| Split | 0.1331 | 0.4744 |
| Joint | 0.1411 | 0.4961 |


**Decision rule:** if the split result is materially worse than the published \(err_I = 0.1331\) and \(err_{R_t} = 0.4744\), stop and diagnose the implementation before running Ghana or either Italy arm. At minimum, check the Case 4 input file, scaling convention, time normalization, random seed, mini-batching, collocation count, epoch settings, and whether the reference implementation’s model/data assumptions have been faithfully reproduced. The course-project validation is not a guarantee that the method will work on Ghana or Italy; it is a necessary implementation check showing that your PyTorch reimplementation behaves plausibly on the published synthetic benchmark.



In [4]:
# Full validation: runs both split and joint against Millevoi Case 4.
!PYTHONPATH=src python src/pinn.py \
    --repo reference \
    --out results/tables/validation_case4.json \
    --mode both \
    --epochs-joint 5000 \
    --epochs-split-data 3000 \
    --epochs-split-ode 1000 \
    --collocation 6000 \
    --seed 34

[joint] err_I=0.1441 (paper 0.1411)  err_Rt=0.4559 (paper 0.4961)  876.6s
[split] err_I=0.1314 (paper 0.1331)  err_Rt=0.4917 (paper 0.4744)  152.9s


For free-tier planning, run the split validation first, since it is substantially cheaper and is the primary gate for the split-first experimental strategy:

In [5]:
# First gate: validate the split formulation before any country-level grid.
!PYTHONPATH=src python src/pinn.py \
    --repo reference \
    --out results/tables/validation_case4.json \
    --mode split \
    --epochs-split-data 3000 \
    --epochs-split-ode 1000 \
    --collocation 6000 \
    --seed 34

[split] err_I=0.1314 (paper 0.1331)  err_Rt=0.4917 (paper 0.4744)  153.9s


Only after this passes should you proceed with Ghana split and Italy-primary split grids. Then run the joint validation before investing in the much more expensive Ghana-joint and Italy-primary-joint experiments:

In [6]:
# Second gate: validate joint before starting any joint country grid.
!PYTHONPATH=src python src/pinn.py \
    --repo reference \
    --out results/tables/validation_case4.json \
    --mode joint \
    --epochs-joint 5000 \
    --collocation 6000 \
    --seed 34

[joint] err_I=0.1441 (paper 0.1411)  err_Rt=0.4559 (paper 0.4961)  849.8s


This two-stage gate preserves the methodological requirement while fitting the split-first, interruption-aware workflow: validate the implementation required for each mode before spending compute on that mode.



## 5. Preprocessing — both countries + the synthetic hospitalisation channel

Runs `prepare_ghana.py` (unchanged) and the new `prepare_italy.py`, then generates:
- `ghana_hospitalisation_synth.csv` (SEIQHRS ablation channel, Dwomoh et al. parameters — explicitly a model-derived proxy, never treated as observed data downstream)
- `italy_primary_national_master.csv` (single-wave, r=6-valid window matched to the Millevoi et al. setup — the clean benchmark used for Objectives 1 & 2)
- `italy_secondary_national_master.csv` (full multi-wave range — Objective 3 regime-shift robustness arm only, kept separate throughout the pipeline)
- `italy_sparseNN.csv` files (20/30/40% controlled random dropout, via `sparsify.py`) generated separately for **both** the primary and secondary series, so downstream grids can pull sparsity regimes from either arm without collapsing the country label

This stage is cheap (<2 min total) and safe to re-run on a free-tier session without worrying about disconnects. Because `prepare_italy.py` now writes two labelled arms instead of one, double-check the `country` column on its outputs is set to `italy_primary`/`italy_secondary` before this feeds into the grid, equity, or stat-test stages — those downstream scripts refuse to run on an ambiguous or missing label.

In [6]:
# NEW step — run this before prepare_italy.py
!PYTHONPATH=src python src/build_italy_national.py --mode bydate \
    --confirmed data/raw/iss_bydate_italy_confirmed.csv \
    --hospitalisations data/raw/iss_bydate_italy_hospitalizations.csv \
    --out data/raw/italy_case_hosp.csv

Wrote data/raw/italy_case_hosp.csv  (999 days, 23,300,510 infections, 684,497 hospitalisations)
            new_infections  new_hospitalisations
date                                            
2020-01-29               2                     0
2020-01-30               0                     0
2020-01-31               0                     0

Sanity check: total infections=23,300,510  total hospitalisations=684,497  ratio=34.04x
Ratio looks plausible for a national COVID-19 series.


Thesis Primary benchmark (for Objectives 1 and 2): a single-wave, r=6-valid window — either Millevoi's original 90 days or a comparable early-2020 window — used for the core split-PINN validation and the Ghana comparability claims. This is your "clean" high-information benchmark.

In [7]:
# PRIMARY arm: single-wave, r=6-valid window (Millevoi-comparable benchmark)
# Used for Objectives 1 & 2 -- core split-PINN validation and Ghana
# comparability claims.
!PYTHONPATH=src python src/prepare_italy.py --arm primary \
    --input data/raw/italy_case_hosp.csv --outdir data/processed

# SECONDARY arm: full multi-wave range, ascertainment correction RESTRICTED
# to its valid window (see prepare_italy.py docstring). Labelled explicitly
# as a regime-shift robustness stress test for Objective 3 only -- NOT/
# presented as a clean data-richness comparison against Ghana.
!PYTHONPATH=src python src/prepare_italy.py --arm secondary --ascertainment restrict \
    --input data/raw/italy_case_hosp.csv --outdir data/processed

import json
print("\n=== PRIMARY arm audit ===")
print(json.dumps(json.load(open("data/processed/italy_primary_audit_summary.json")), indent=2))
print("\n=== SECONDARY arm audit ===")
print(json.dumps(json.load(open("data/processed/italy_secondary_audit_summary.json")), indent=2))

{
  "arm": "primary",
  "date_start": "2020-02-20",
  "date_end": "2020-05-20",
  "n_days": 91,
  "total_cases_raw": 228782.0,
  "total_cases_corrected": 6824622.0,
  "total_hospitalisations": 87304.0,
  "primary_infectious_period_D": 5,
  "n_weeks": 13,
  "ascertainment_mode": "fixed",
  "ratio": 6.0,
  "warning": "Flat r=6 applied across full range -- valid only for the 2020 wave-1 window; interpret results outside that window with caution."
}
{
  "arm": "secondary",
  "date_start": "2020-01-29",
  "date_end": "2022-10-23",
  "n_days": 999,
  "total_cases_raw": 23300510.0,
  "total_cases_corrected": 8148840.0,
  "total_hospitalisations": 684497.0,
  "primary_infectious_period_D": 5,
  "n_weeks": 143,
  "ascertainment_mode": "restrict",
  "ratio": 6.0,
  "valid_until": "2020-08-31",
  "note": "Corrected infections are NaN beyond the r=6-valid window; raw new_infections retained for that period instead. Use raw counts, not corrected, past this date."
}

=== PRIMARY arm audit ===
{
  "a

In [8]:

!PYTHONPATH=src python src/prepare_ghana.py \
    --input data/raw/ghana_hera.csv --outdir data/processed

!PYTHONPATH=src python src/prepare_italy.py \
    --input data/raw/italy_case_hosp.csv --outdir data/processed

!PYTHONPATH=src python src/synth_hospitalisation.py \
    --master data/processed/ghana_national_master.csv \
    --params dwomoh_scenario5 --sensitivity \
    --out data/processed/ghana_hospitalisation_synth.csv

!PYTHONPATH=src python src/sparsify.py \
    --master data/processed/italy_primary_national_master.csv \
    --pct 20 30 40 \
    --outdir data/processed/italy_synth_sparse

# Merge sparsify's per-file observed-columns back onto one Italy master file
import pandas as pd
master = pd.read_csv("data/processed/italy_primary_national_master.csv", index_col=0, parse_dates=True)
for pct in (20, 30, 40):
    s = pd.read_csv(f"data/processed/italy_synth_sparse/italy_sparse{pct}.csv", index_col=0, parse_dates=True)
    master[f"observed_sparse{pct}"] = s[f"observed_sparse{pct}"]
master.to_csv("data/processed/italy_primary_national_master.csv")
print("Italy master updated with sparsity columns:",
      [c for c in master.columns if c.startswith("observed_")])


{
  "date_start": "2020-03-12",
  "date_end": "2021-05-19",
  "n_days": 434,
  "n_regions": 17,
  "total_cases": 93674,
  "total_deaths": 783,
  "total_recoveries": 91647,
  "observed_days_simple": 317,
  "observed_pct_simple": 73.0,
  "observed_days_context": 330,
  "unreported_days": 117,
  "longest_gap": 14,
  "n_gaps": 70,
  "negative_records_clipped": 8,
  "cases_excl_unspecified": 91946,
  "primary_infectious_period_D": 5,
  "n_weeks": 62
}

Wrote:
  audit_summary.json
  ghana_hospitalisation_synth.csv
  ghana_hospitalisation_synth.meta.json
  ghana_millevoi_format.csv
  ghana_national_excl_unspecified.csv
  ghana_national_master.csv
  ghana_weekly.csv
  italy_primary_audit_summary.json
  italy_primary_millevoi_format.csv
  italy_primary_national_master.csv
  italy_primary_weekly.csv
  italy_secondary_audit_summary.json
  italy_secondary_millevoi_format.csv
  italy_secondary_national_master.csv
  italy_secondary_weekly.csv
  italy_synth_sparse
  revision_log.csv
{
  "arm": "prima

## 6. The grids — staged and sharded for free-tier survivability

### 6.1 Sizing (what to expect)

| Arm | Fits | Rough wall-clock, 2 vCPU (free) | 8 vCPU | 16 vCPU |
|---|---|---|---|---|
| Ghana split (unchanged) | 54 | ~100 min | ~25 min | ~13 min |
| Ghana joint (unchanged) | 54 | ~13 h | ~3.3 h | ~1.7 h |
| Italy PRIMARY split (4 regimes: daily, weekly, sparse20, sparse30) | 90 | ~90 min | ~22 min | ~11 min |
| Italy PRIMARY joint (4 regimes) | 90 | ~11 h | ~2.8 h | ~1.4 h |
| Italy SECONDARY split (4 regimes, multi-wave, optional Obj. 3 robustness) | 150 | ~150 min | ~37 min | ~19 min |
| Italy SECONDARY joint (4 regimes, optional) | 150 | ~18 h | ~4.6 h | ~2.3 h |
| Hospitalisation ablation (Ghana, split only, ×2 for with/without H) | 54 | ~100 min | ~25 min | ~13 min |

**On free tier (2 vCPU, no background execution): run split arms only in any one sitting.** Joint arms must go on a paid/High-RAM session with background execution, or be broken into per-origin chunks across several free sessions (Section 6.4). The SECONDARY Italy arm is larger and longer-running because it spans multiple epidemic waves rather than one; since it feeds only the Objective 3 robustness check (not the core equity comparison), it is lowest priority — run it last, and only once PRIMARY is done and validated.

### 6.2 Split-first policy (unchanged principle, now for three arms)

Always run every `split` arm before any `joint` arm — split is ~6× cheaper and gives you a working results table even if the session dies before joint finishes. With three arms in play, the recommended order is:

1. Ghana split → Italy PRIMARY split (core comparison, cheapest first).
2. Ghana joint → Italy PRIMARY joint (once split results confirm the pipeline works).
3. Italy SECONDARY split, then joint — only after PRIMARY is complete, since SECONDARY is an optional Objective 3 robustness arm, not required for the main equity result.

This ordering ensures a session dying early still leaves you with the two arms your central equity claim depends on (Ghana + Italy PRIMARY), rather than burning free-tier hours on the robustness-only SECONDARY arm before the core comparison is even finished.

In [10]:
import os
NW = os.cpu_count()
print(f"Using {NW} workers.")

# --- Ghana split (unchanged grid) ---
!PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_parallel.py \
    --workers {NW} --modes split \
    --origins 45 60 75 90 \
    --regimes daily weekly mask \
    --seeds 1 2 3 --horizon 15 \
    --out results/tables/grid_ghana_split.csv

# # --- Italy split (new grid) ---
# !PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_italy.py \
#     --out results/tables/italy_grid_split.csv \
#     --origins 60 75 90 105 --horizon 15 \
#     --regimes daily weekly sparse20 sparse30 sparse40 \
#     --modes split --seeds 1 2 3


# --- Italy PRIMARY arm split grid (single-wave, Millevoi-comparable) ---
!PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_italy.py \
    --master data/processed/italy_primary_national_master.csv \
    --out results/tables/italy_primary_grid_split.csv \
    --origins 45 60 75 --horizon 15 \
    --regimes daily weekly sparse20 sparse30 sparse40 \
    --modes split --seeds 1 2 3



Using 4 workers.


36 fits (0 joint) across 4 workers (FULL epochs)
estimated wall-clock: 0.5 h
[   4/36] o= 45 mask   pinn_split s=1 MASE=   7.76  ETA  29.1 min
[   5/36] o= 45 daily  pinn_split s=3 MASE=   6.27  ETA  23.9 min
[   6/36] o= 45 daily  pinn_split s=2 MASE=   6.35  ETA  19.3 min
[   7/36] o= 45 daily  pinn_split s=1 MASE=   6.38  ETA  16.0 min
[   8/36] o= 45 mask   pinn_split s=2 MASE=   7.74  ETA  25.2 min
[   9/36] o= 45 mask   pinn_split s=3 MASE=   7.92  ETA  22.3 min
[  13/36] o= 60 daily  pinn_split s=2 MASE=   8.08  ETA  13.7 min
[  14/36] o= 60 daily  pinn_split s=1 MASE=   8.06  ETA  12.2 min
[  15/36] o= 60 mask   pinn_split s=1 MASE=  10.58  ETA  15.4 min
[  16/36] o= 60 daily  pinn_split s=3 MASE=   8.19  ETA  13.8 min
[  17/36] o= 60 mask   pinn_split s=2 MASE=  10.57  ETA  12.7 min
[  18/36] o= 60 mask   pinn_split s=3 MASE=  10.81  ETA  11.4 min
[  19/36] o= 75 weekly pinn_split s=1 MASE=   1.53  ETA  13.4 min
[  20/36] o= 75 daily  pinn_split s=1 MASE=   1.23  ETA  12.1 min

# Objective 3 supplementary robustness read (regime-shift stress test).
# NOT part of the primary equity/comparability claim above.

In [9]:
 !PYTHONPATH=src python src/sparsify.py \
    --master data/processed/italy_secondary_national_master.csv \
    --country italy_secondary \
    --pct 20 30 40 \
    --inplace

usage: sparsify.py [-h] --master MASTER [--pct PCT [PCT ...]] --outdir OUTDIR
sparsify.py: error: the following arguments are required: --outdir


In [4]:
# --- Italy SECONDARY arm split grid (full range, regime-shift stress test) ---
!PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_italy.py \
    --master data/processed/italy_secondary_national_master.csv \
    --out results/tables/italy_secondary_grid_split.csv \
    --origins 75 90 100 150 300 --horizon 15 \
    --regimes daily weekly sparse20 sparse30 sparse40 \
    --modes split --seeds 1 2 3


  origin= 75 daily     split seed=1 MASE=  10.263
  origin= 75 daily     split seed=2 MASE=  11.538
  origin= 75 daily     split seed=3 MASE=   9.211
  origin= 75 weekly    split seed=1 MASE=   9.895
  origin= 75 weekly    split seed=2 MASE=   3.986
  origin= 75 weekly    split seed=3 MASE=   5.089
Traceback (most recent call last):
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 131, in <module>
    main()
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 124, in main
    df = run(master, cfg, a.origins, a.horizon, a.regimes, a.modes, a.seeds, a.out)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 71, in run
    tr_idx = regime_indices_italy(master, regime, origin)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", li

### 6.3 Joint arms (paid tier / background execution recommended)

Enable **Runtime → Background execution** first if available, or the joint
runs below will die when the tab closes. If you are on free tier without
background execution, use the chunked plan in 6.4 instead of this cell.

In [5]:
!PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_parallel.py \
    --workers {NW} --modes joint \
    --origins 45 60 75 90 \
    --regimes daily weekly mask \
    --seeds 1 2 3 --horizon 15 \
    --out results/tables/grid_ghana_joint.csv

# !PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_italy.py \
#     --out results/tables/italy_grid_joint.csv \
#     --origins 60 75 90 105 --horizon 15 \
#     --regimes daily weekly sparse20 sparse30 sparse40 \
#     --modes joint --seeds 1 2 3


!PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_italy.py \
    --master data/processed/italy_primary_national_master.csv \
    --out results/tables/italy_primary_grid_joint.csv \
    --origins 45 60 75 --horizon 15 \
    --regimes daily weekly sparse20 sparse30 sparse40 \
    --modes joint --seeds 1 2 3


usage: run_parallel.py [-h] [--master MASTER] [--out OUT] [--workers WORKERS]
                       [--origins ORIGINS [ORIGINS ...]]
                       [--regimes REGIMES [REGIMES ...]]
                       [--modes MODES [MODES ...]] [--seeds SEEDS [SEEDS ...]]
                       [--horizon HORIZON] [--quick] [--shard SHARD] [--of OF]
run_parallel.py: error: argument --workers: invalid int value: '{NW}'
Traceback (most recent call last):
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 131, in <module>
    main()
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 124, in main
    df = run(master, cfg, a.origins, a.horizon, a.regimes, a.modes, a.seeds, a.out)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 85, in run
    m.fit_joint(t_data, I_obs)
  File "/workspaces/PINN-Thesis-Framework-MPh

In [ ]:

!PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_italy.py \
    --master data/processed/italy_secondary_national_master.csv \
    --out results/tables/italy_secondary_grid_joint.csv \
    --origins 75 90 100 150 --horizon 15 \
    --regimes daily weekly sparse20 sparse30 sparse40 \
    --modes joint --seeds 1 2 3

  origin= 75 daily     joint seed=1 MASE=   6.921
  origin= 75 daily     joint seed=2 MASE=   6.847
  origin= 75 daily     joint seed=3 MASE=   6.690
  origin= 75 weekly    joint seed=1 MASE=   1.792
  origin= 75 weekly    joint seed=2 MASE=  10.338
  origin= 75 weekly    joint seed=3 MASE=   1.247
Traceback (most recent call last):
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 131, in <module>
    main()
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 124, in main
    df = run(master, cfg, a.origins, a.horizon, a.regimes, a.modes, a.seeds, a.out)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", line 71, in run
    tr_idx = regime_indices_italy(master, regime, origin)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/run_italy.py", li

In [14]:
# First session: Ghana split fits only.
# --max-fits 6 means it finishes after six new fits and leaves a valid checkpoint.
!PYTHONPATH=src python src/checkpoint_grid_runner.py \
    --master data/processed/ghana_national_master.csv \
    --out results/tables/grid_ghana.csv \
    --origins 60 75 \
    --regimes daily weekly mask \
    --modes split \
    --seeds 1 2 3 \
    --max-fits 6

!PYTHONPATH=src python src/checkpoint_grid_runner.py \
    --master data/processed/italy_primary_national_master.csv \
    --out results/tables/grid_italy.csv \
    --origins 60 75 \
    --regimes daily weekly \
    --modes split \
    --seeds 1 2 3 \
    --max-fits 6

Checkpoint: results/tables/grid_ghana.csv
Completed PINN fits: 0 | Remaining: 18
SAVED   1 | origin= 60 daily     split seed=1 MASE=8.060 (153.8s)
SAVED   2 | origin= 60 daily     split seed=2 MASE=8.080 (153.4s)
SAVED   3 | origin= 60 daily     split seed=3 MASE=8.190 (154.7s)
SKIP origin=60 regime=weekly: fewer than 10 training points
SKIP origin=60 regime=weekly: fewer than 10 training points
SKIP origin=60 regime=weekly: fewer than 10 training points
SAVED   4 | origin= 60 mask      split seed=1 MASE=10.582 (138.3s)
SAVED   5 | origin= 60 mask      split seed=2 MASE=10.569 (138.3s)
SAVED   6 | origin= 60 mask      split seed=3 MASE=10.814 (136.6s)
Finished this session: 6 new fit(s); remaining from this invocation: 12
Checkpoint is current: results/tables/grid_ghana.csv
Checkpoint: results/tables/grid_italy.csv
Completed PINN fits: 0 | Remaining: 12
SAVED   1 | origin= 60 daily     split seed=1 MASE=2.649 (154.5s)
SAVED   2 | origin= 60 daily     split seed=2 MASE=2.117 (153.9s)
SA

In [4]:
# Resume after a disconnect or in the next scheduled session.
!PYTHONPATH=src python src/checkpoint_grid_runner.py \
    --master data/processed/ghana_national_master.csv \
    --out results/tables/grid_ghana.csv \
    --origins 60 75 90 105 \
    --regimes daily weekly mask \
    --modes split \
    --seeds 1 2 3 \
    --max-fits 6


# Resume after a disconnect or in the next scheduled session.
!PYTHONPATH=src python src/checkpoint_grid_runner.py \
    --master data/processed/italy_primary_national_master.csv \
    --out results/tables/grid_italy.csv \
    --origins 60 75 \
    --regimes daily weekly \
    --modes split \
    --seeds 1 2 3 \
    --max-fits 6

Checkpoint: results/tables/grid_ghana.csv
Completed PINN fits: 6 | Remaining: 30
SKIP origin=60 regime=weekly: fewer than 10 training points
SKIP origin=60 regime=weekly: fewer than 10 training points
SKIP origin=60 regime=weekly: fewer than 10 training points
SAVED   1 | origin= 75 daily     split seed=1 MASE=1.230 (161.0s)
SAVED   2 | origin= 75 daily     split seed=2 MASE=2.030 (153.6s)
SAVED   3 | origin= 75 daily     split seed=3 MASE=1.273 (153.1s)
Traceback (most recent call last):
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/checkpoint_grid_runner.py", line 193, in <module>
    main()
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/checkpoint_grid_runner.py", line 169, in main
    model.fit_split(t_data, I_obs)
  File "/workspaces/PINN-Thesis-Framework-MPhil-2026-2027-/src/pinn.py", line 235, in fit_split
    l2 = self._run_minibatch(list(self.R_net.parameters()),
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/PI

In [ ]:
# Ghana joint fits: use the SAME output CSV.
# Completed split rows are retained; only joint rows are added.
!PYTHONPATH=src python src/checkpoint_grid_runner.py \
    --master data/processed/ghana_national_master.csv \
    --out results/tables/grid_ghana.csv \
    --origins 60 75 90 105 \
    --regimes daily weekly mask \
    --modes joint \
    --seeds 1 2 3 \
    --max-fits 2


# Italy joint fits: use the SAME output CSV.
# Completed split rows are retained; only joint rows are added.
!PYTHONPATH=src python src/checkpoint_grid_runner.py \
    --master data/processed/italy_primary_national_master.csv \
    --out results/tables/grid_italy.csv \
    --origins 60 75 \
    --regimes daily weekly \
    --modes joint \
    --seeds 1 2 3 \
    --max-fits 2

Checkpoint: results/tables/grid_ghana.csv
Completed PINN fits: 9 | Remaining: 36
SAVED   1 | origin= 60 daily     joint seed=1 MASE=14.776 (939.2s)
SAVED   2 | origin= 60 daily     joint seed=2 MASE=13.666 (948.3s)
Finished this session: 2 new fit(s); remaining from this invocation: 34
Checkpoint is current: results/tables/grid_ghana.csv
Checkpoint: results/tables/grid_italy.csv
Completed PINN fits: 9 | Remaining: 12
SAVED   1 | origin= 60 daily     joint seed=1 MASE=3.355 (947.2s)


### 6.4 Free-tier fallback: chunk the joint grid by origin across sessions

If background execution is unavailable, split the joint grid by origin and
run one or two origins per session, appending to the same CSV. Re-run this
cell each session with the NEXT_ORIGINS list updated; already-completed
origins are skipped automatically via the `--resume` check against existing
rows in the output CSV (see `run_stages.py`'s resume-safe pattern, reused
here for a single origin at a time).

In [ ]:
# Example: session 1 covers origins 100, 140 only. Update NEXT_ORIGINS next session.
NEXT_ORIGINS = [60, 75]

!PYTHONPATH=src OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python src/run_parallel.py \
    --workers {NW} --modes joint \
    --origins {" ".join(map(str, NEXT_ORIGINS))} \
    --regimes daily weekly mask \
    --seeds 1 2 3 --horizon 15 \
    --out results/tables/grid_ghana_joint_chunk_{"_".join(map(str, NEXT_ORIGINS))}.csv

# After all chunks are done, concatenate:
# import pandas as pd, glob
# chunks = [pd.read_csv(f) for f in glob.glob("results/tables/grid_ghana_joint_chunk_*.csv")]
# pd.concat(chunks, ignore_index=True).to_csv("results/tables/grid_ghana_joint.csv", index=False)


### 6.5 Hospitalisation ablation (Ghana split only — cheap, safe for free tier)

Compares infection-only training against infection+synthetic-H training,
using the SEIQHRS-derived channel from Stage 5. This directly tests whether
a second observed channel helps the way it does for Italy's real
hospitalisation series in Millevoi et al.'s Cases 6–7.

**Note:** `run_ghana.py`/`run_parallel.py` as inherited only support a
single-channel `I_proxy` target. Running the two-channel ablation requires
either (a) extending `run_ghana.py` with an `--aux-channel` flag pointing at
`H_synthetic_seiqhrs`, mirroring how Millevoi et al. add hospitalisations as
a second loss term, or (b) a standalone ablation script. This is flagged
here as **implementation work still to be done on `pinn.py`'s loss function**
before this stage can run — it is not yet wired into the shared CLI.

In [ ]:
# Placeholder — wire up once pinn.py exposes a two-channel loss (aux_channel arg).
import pandas as pd
ghana_master = pd.read_csv("data/processed/ghana_national_master.csv", index_col=0, parse_dates=True)
h_synth = pd.read_csv("data/processed/ghana_hospitalisation_synth.csv", index_col=0, parse_dates=True)
ghana_master["H_synthetic_seiqhrs"] = h_synth["H_synthetic_seiqhrs"]
ghana_master.to_csv("data/processed/ghana_national_master_with_H.csv")

!PYTHONPATH=src python src/run_ghana.py --aux-channel H_synthetic_seiqhrs \
    --modes split --origins 60 75 90 105 --seeds 1 2 3 \
    --out results/tables/grid_ghana_split_with_H.csv
print("Ablation stage is a placeholder pending pinn.py two-channel support. See markdown above.")


## 7. Merge grids, then equity metrics and statistical tests

In [ ]:
# import pandas as pd

# ghana_split = pd.read_csv("results/tables/grid_ghana_split.csv")
# italy_split = pd.read_csv("results/tables/italy_grid_split.csv")

# import os
# ghana_parts = [ghana_split]
# if os.path.exists("results/tables/grid_ghana_joint.csv"):
#     ghana_parts.append(pd.read_csv("results/tables/grid_ghana_joint.csv"))
# italy_parts = [italy_split]
# if os.path.exists("results/tables/italy_grid_joint.csv"):
#     italy_parts.append(pd.read_csv("results/tables/italy_grid_joint.csv"))

# ghana_grid = pd.concat(ghana_parts, ignore_index=True)
# italy_grid = pd.concat(italy_parts, ignore_index=True)
# ghana_grid.to_csv("results/tables/grid.csv", index=False)
# italy_grid.to_csv("results/tables/italy_grid.csv", index=False)
# print("Ghana grid:", ghana_grid.shape, " Italy grid:", italy_grid.shape)


In [ ]:
import pandas as pd, os

def load_parts(paths):
    parts = [pd.read_csv(p) for p in paths if os.path.exists(p)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

ghana_grid = load_parts(["results/tables/grid_ghana_split.csv",
                          "results/tables/grid_ghana_joint.csv"])
ghana_grid["country"] = "ghana"
ghana_grid.to_csv("results/tables/grid.csv", index=False)

italy_primary = load_parts(["results/tables/italy_primary_grid_split.csv",
                             "results/tables/italy_primary_grid_joint.csv"])
italy_primary["country"] = "italy_primary"

italy_secondary = load_parts(["results/tables/italy_secondary_grid_split.csv",
                               "results/tables/italy_secondary_grid_joint.csv"])
italy_secondary["country"] = "italy_secondary"

# Objectives 1 & 2 (clean comparability) use italy_primary ONLY against Ghana.
italy_grid_for_equity = italy_primary.copy()
italy_grid_for_equity.to_csv("results/tables/italy_grid.csv", index=False)



In [ ]:
# Equity metrics: main Ghana vs Italy-primary comparison
!PYTHONPATH=src python src/equity_metrics.py \
    --grid results/tables/grid_ghana.csv \
    --italy-grid results/tables/grid_italy.csv \
    --out results/tables/equity_summary_primary.csv

In [ ]:
# Statistical tests: main Ghana vs Italy-primary comparison
!PYTHONPATH=src python src/stat_tests.py \
    --grid results/tables/grid_ghana.csv \
    --italy-grid results/tables/grid_italy.csv \
    --out results/tables/significance_primary.csv \
    --italy-arm primary

## 8. Rt validity — both countries (extends `extract_rt.py`)

Ghana: checked against Dwomoh et al.'s Re estimates (unchanged from course
project). Italy: checked against the ISS regional/national Rt series
(`iss_rt_*.csv`) uploaded alongside the raw data.

In [ ]:
# Ghana: full Ghana analysis series
!PYTHONPATH=src python src/extract_rt.py \
    --master data/processed/ghana_national_master.csv \
    --country ghana \
    --outdir results/rt

# Italy PRIMARY: single-wave, main comparison arm
!PYTHONPATH=src python src/extract_rt.py \
    --master data/processed/italy_primary_national_master.csv \
    --country italy_primary \
    --outdir results/rt



## 9. Two-country data-quality figure and degradation plots

In [16]:
!PYTHONPATH=src python src/fig_dataquality.py


ok: wrote results/figures/fig_data_quality.png


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

r = pd.read_csv("results/tables/grid.csv"); r["country"] = "ghana"
ri = pd.read_csv("results/tables/italy_grid.csv")
if "country" not in ri.columns: ri["country"] = "italy"
combined = pd.concat([r, ri], ignore_index=True)
pinn = combined[combined.method.str.startswith("pinn")]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for country, marker in [("ghana", "o"), ("italy", "s")]:
    sub = pinn[(pinn.country == country) & (pinn.method == "pinn_split")]
    order = [g for g in ["daily", "weekly", "mask", "sparse20", "sparse30", "sparse40"] if g in sub.regime.unique()]
    med = [sub[sub.regime == g].mase.median() for g in order]
    ax[0].plot(order, med, marker=marker, label=f"{country} (split)")
ax[0].set_ylabel("Median MASE"); ax[0].set_xlabel("Observation regime")
ax[0].set_title("Split-schedule degradation, Italy vs Ghana"); ax[0].legend(frameon=False)
ax[0].tick_params(axis="x", rotation=30)

eq = pd.read_csv("results/tables/equity_summary.csv")
piv = eq.groupby(["country", "regime"])["mase_gap_pct"].median().unstack("country")
piv.plot(kind="bar", ax=ax[1])
ax[1].set_ylabel("Median MASE gap vs daily (%)"); ax[1].set_title("Equity gap by regime")
for a_ in ax: a_.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("results/figures/fig_italy_ghana_equity.png", dpi=180); plt.show()


# Objective 3 supplementary robustness read (regime-shift stress test).
# NOT part of the primary equity/comparability claim above.

In [ ]:
# Objective 3 robustness supplement keeps italy_secondary separate, never
# silently pooled with the primary/Ghana comparability claim.
italy_secondary.to_csv("results/tables/italy_grid_secondary_robustness.csv", index=False)

print("Ghana:", ghana_grid.shape, " Italy primary:", italy_primary.shape,
      " Italy secondary (robustness-only):", italy_secondary.shape)

In [ ]:
# Equity: also generate a separate Italy-secondary robustness output
!PYTHONPATH=src python src/equity_metrics.py \
    --grid results/tables/grid_ghana.csv \
    --italy-grid results/tables/grid_italy.csv \
    --out results/tables/equity_summary_primary.csv \
    --include-secondary

In [ ]:
# Objective 3 supplementary robustness read (regime-shift stress test).
# NOT part of the primary equity/comparability claim above.
sec = pd.read_csv("results/tables/italy_grid_secondary_robustness.csv")
sec_pinn = sec[sec.method.str.startswith("pinn")]
print(sec_pinn.groupby(["method", "regime"])["mase"].median().round(3).to_string())
print("\nNote: italy_secondary spans multiple COVID-19 waves/variants and a "
      "restricted (partially NaN) ascertainment correction -- interpret as "
      "a regime-shift robustness signal, not a like-for-like data-richness "
      "comparison against Ghana or italy_primary.")

In [ ]:
# Italy SECONDARY: optional multi-wave Objective-3 robustness arm
!PYTHONPATH=src python src/extract_rt.py \
    --master data/processed/italy_secondary_national_master.csv \
    --country italy_secondary \
    --outdir results/rt

## 10. Download everything

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("thesis_results", "zip", "results")
files.download("thesis_results.zip")
